在 SNN 中，神经元通过离散的脉冲 (Spike = 0 或 1) 进行通信，这引入了时间动力学。为了模拟这一过程，我们需要数学模型来描述膜电位 (Voltage) 的变化以及突触电流的产生。

## 神经元模型

### 1. LIF 神经元模型 (Leaky Integrate-and-Fire)

LIF 是最基础的“漏电积分触发”模型。可以将其比作一个底部有洞的水桶：

数学公式：

$$\frac{dV}{dt} = -\frac{V - V_{reset}}{\tau} + \frac{I}{c_m}$$

变量解释：

$V$: 神经元的膜电位 (Membrane Voltage)，即“水桶里的水位”。

$V_{reset}$: 重置电位 (静息电位)，没有输入时电位倾向于恢复到的基准值。

$\tau$: 膜时间常数 (Membrane time constant)，决定了漏电的速度。$\tau$ 越大，漏电越慢，记忆保持越久。

$I$: 输入的突触后电流 (Input current)，即“注入水桶的水”。$c_m$: 膜电容 (Membrane capacitance)，调节电流对电位变化的影响幅度。

触发机制 (Fire) 与 不应期 (Refractory Period)：当 $V$ 达到阈值 ($V_{threshold}$) 时，发放一个 Spike，随后 $V$ 瞬间降至 $V_{reset}$。如果设定了 $\tau_{ref}$，神经元会在 $\tau_{ref}$ 毫秒内进入“不应期”，期间不接收输入也不发放脉冲 (If $\tau_{ref}$ is specified, a refractory period prevents spiking for $\tau_{ref}$ milliseconds after each spike)。

### 2. GLIF 神经元模型 (Generalized LIF)

真实的生物神经元具有更丰富的多时间尺度动态（如适应性放电）。GLIF 引入了内部状态变量 $I_{asc}$ (Ascending currents) 来模拟复杂的离子通道行为。

数学公式：

$$\frac{dV}{dt} = -\frac{V - V_{rest}}{\tau} + \frac{I_{in} + \sum I_{asc}}{c_m}$$
$$\frac{dI_{asc}}{dt} = -k \cdot I_{asc}$$

发放脉冲时 (At spike):$$I_{asc} \mathrel{+}= asc\_amps$$

变量解释补充：

$I_{in}$: 外部输入的总电流。

$I_{asc}$: 上升电流，GLIF 可以有多个不同时间尺度的上升电流共同作用。

$k$: $I_{asc}$ 的衰减速率 (Decay rates)。

$asc\_amps$: 神经元每次发放 Spike 时，$I_{asc}$ 发生阶跃变化的幅度。这使得神经元的放电历史会影响其未来的兴奋性。

## 突触模型

在脉冲神经网络中，上游神经元发放的脉冲（Spike）是一个瞬时的离散事件（瞬时值为 1，其余时刻为 0）。然而，在真实的生物神经系统中，信号并不会瞬间跨越突触。脉冲到达突触前膜后，会促使神经递质释放，进而使得下游神经元的离子通道打开，产生一段连续变化的突触后电流 (Post-Synaptic Current, PSC)。
突触模型的核心作用，就是完成这种**“离散事件”到“连续电流”的平滑转换**。

### Alpha突触模型

Alpha 模型是神经科学中最经典且具有强生物学基础的突触模型之一（对应 btorch 中的 synapse.AlphaPSC）。当接收到一个输入脉冲时，它会产生一个先快速上升、随后缓慢衰减的电流轨迹。

数学公式：假设上游神经元在 $t=0$ 时刻发放了一个脉冲，下游神经元接收到的突触后电流 $I(t)$ 可表示为：

$$I(t) = I_{max} \cdot \frac{t}{\tau_{syn}} e^{1 - \frac{t}{\tau_{syn}}} \quad (t \ge 0)$$
变量解释：

$I(t)$: $t$ 时刻的突触后电流大小。

$I_{max}$: 电流响应的峰值（最大振幅），在神经网络中，这个值通常与两个神经元之间的连接权重 (Weight) 直接相关。

$\tau_{syn}$: 突触时间常数 (Synaptic time constant)。这是 Alpha 模型中最核心的参数。

$e$: 自然常数（欧拉数，约 2.718）。

物理动态特性：响应延迟与上升：脉冲到达后（$t>0$），递质开始作用，电流迅速上升。到达峰值：当时间 $t = \tau_{syn}$ 时，公式中的导数为零，电流精确地达到最高峰 $I_{max}$。指数衰减：越过峰值后（$t > \tau_{syn}$），随着神经递质被清除，电流开始缓慢下降并趋近于零。$\tau_{syn}$ 的值越大，这个“拖尾”的衰减过程就越漫长，意味着当前脉冲对下游神经元的影响会持续更久。

This tutorial walks through building a minimal Recurrent Spiking Neural Network (RSNN) with btorch.

A minimal RSNN needs three things:

Neuron model — defines how membrane voltage evolves and when spikes fire.
Synapse model — transforms spikes into post-synaptic currents.
Connection layer — defines the weight matrix between neurons.

In [ ]:
import torch
import torch.nn as nn
from btorch.models import environ, functional, glif, rnn, synapse
from btorch.models.linear import DenseConn


class MinimalRSNN(nn.Module):
    def __init__(self, num_input, num_hidden, num_output, device="cpu"):
        super().__init__()

        # 1. Input projection
        self.fc_in = nn.Linear(num_input, num_hidden, bias=False, device=device)

        # 2. Spiking neuron
        neuron_module = glif.GLIF3(
            n_neuron=num_hidden,
            v_threshold=-45.0, #发放脉冲的阈值电压 (-45.0 mV)。
            v_reset=-60.0, #发放脉冲后的重置电压，同时也常作为公式中的 V_rest (-60.0 mV)。
            c_m=2.0, #膜电容 (2.0)。
            tau=20.0, #膜时间常数，控制膜电位的自然衰减速度 (20.0 ms)。
            tau_ref=2.0, #绝对不应期，发放脉冲后罢工的时间 (2.0 ms)。
            k=[0.1, 0.2], #（GLIF特有）定义了两种不同时间尺度的 I_asc 电流，它们随时间自然衰减的速率分别为 0.1 和 0.2。
            asc_amps=[1.0, -2.0], #（GLIF特有）每次神经元发放脉冲 (At spike) 时，第一种电流增加 1.0，第二种电流减少 2.0 (即 += asc_amps)。这能模拟神经元在连续放电后的疲劳或兴奋等复杂的生物学行为。
            step_mode="s",   # 表示按照单个时间步处理
            backend="torch",
            device=device,
        )

        # 3. Recurrent connection
        # 定义隐藏层神经元之间的循环连接权重矩阵
        conn = DenseConn(num_hidden, num_hidden, bias=None, device=device)

        # 4. Synapse
        psc_module = synapse.AlphaPSC(
            n_neuron=num_hidden,
            tau_syn=5.0,
            linear=conn,
            step_mode="s",
        )

        # 5. Recurrent wrapper (multi-step)
        self.brain = rnn.RecurrentNN(
            neuron=neuron_module,
            synapse=psc_module,
            step_mode="m",
            update_state_names=("neuron.v", "synapse.psc"),
        )

        # 6. Output readout
        self.fc_out = nn.Linear(num_hidden, num_output, bias=False, device=device)

    def forward(self, x):
        x = self.fc_in(x)                 # (T, Batch, num_input) -> (T, Batch, N)
        spike, states = self.brain(x)     # spike: (T, Batch, N)
        rate = spike.mean(dim=0)          # (Batch, N)
        out = self.fc_out(rate)           # (Batch, num_output)
        return out

### Initializing State
Before the first forward pass, call init_net_state to register and initialize memory buffers:

### Running a Forward Pass
Wrap the forward pass in an environ.context(dt=...) block:

In [11]:
# Keep dtype/device consistent across all params, buffers, states, and inputs
torch.set_default_dtype(torch.float32)
DTYPE = torch.float32
DEVICE = torch.device("cpu")

environ.set(dt=1.0)

# Rebuild model on explicit dtype/device to avoid hidden mixed dtypes in nested modules
model = MinimalRSNN(num_input=20, num_hidden=64, num_output=5, device=str(DEVICE))
model = model.to(device=DEVICE, dtype=DTYPE)

# Initialize/reset states AFTER dtype/device are finalized
functional.init_net_state(model, batch_size=4, device=str(DEVICE))
functional.reset_net(model, batch_size=4)

# Some btorch/spikingjelly memory states are recreated from Python floats during
# reset and may become float64. Force all floating buffers back to desired dtype.
def _cast_all_floating_buffers_(module: nn.Module, dtype: torch.dtype, device: torch.device):
    for buffer_name, buffer in list(module.named_buffers()):
        if buffer is not None and torch.is_floating_point(buffer):
            casted = buffer.to(device=device, dtype=dtype)
            # buffer_name can be nested like "brain.synapse.psc"
            parts = buffer_name.split(".")
            target = module
            for p in parts[:-1]:
                target = getattr(target, p)
            setattr(target, parts[-1], casted)

_cast_all_floating_buffers_(model, DTYPE, DEVICE)

# Create input with explicit dtype/device
inputs = torch.rand((100, 4, 20), device=DEVICE, dtype=DTYPE)  # (T, Batch, input_dim)

# Check both visible and nested recurrent/synapse dtypes
print("inputs:", inputs.dtype, inputs.device)
print("fc_in.weight:", model.fc_in.weight.dtype, model.fc_in.weight.device)
print("synapse.linear.weight:", model.brain.synapse.linear.weight.dtype, model.brain.synapse.linear.weight.device)
print("synapse.psc state:", model.brain.synapse.psc.dtype, model.brain.synapse.psc.device)

with environ.context(dt=1.0):
    out = model(inputs)  # (Batch, num_output)

print("out:", out.dtype, out.device, out.shape)

inputs: torch.float32 cpu
fc_in.weight: torch.float32 cpu
synapse.linear.weight: torch.float32 cpu
synapse.psc state: torch.float32 cpu
out: torch.float32 cpu torch.Size([4, 5])


### Inspecting Recorded States
update_state_names tells RecurrentNN which variables to save. The returned states dict uses dot notation:

In [12]:
####
inputs = torch.rand((100, 4, 64), device=DEVICE, dtype=DTYPE)  # (T, Batch, input_dim)
with environ.context(dt=1.0):
    spike, states = model.brain(inputs)

print(states["neuron.v"].shape)     # (T, Batch, N)
print(states["synapse.psc"].shape)  # (T, Batch, N)

torch.Size([100, 4, 64])
torch.Size([100, 4, 64])


You can unflatten the dict for easier access:

In [13]:
from btorch.utils.dict_utils import unflatten_dict
nested = unflatten_dict(states, dot=True)
nested["neuron"]["v"][:, 0, :]   # voltage of batch 0

tensor([[-61.0989, -59.1580, -62.5523,  ..., -62.3810, -58.2102, -51.5261],
        [-60.7111, -58.7798, -62.2351,  ..., -62.2483, -57.8688, -51.7511],
        [-60.6137, -58.7169, -61.9049,  ..., -61.6939, -57.7200, -51.8256],
        ...,
        [-54.3362, -54.6143, -55.2800,  ..., -54.7361, -54.9679, -55.8008],
        [-54.4783, -54.4367, -55.1024,  ..., -54.7409, -55.0133, -55.5667],
        [-54.7366, -54.3310, -55.2748,  ..., -54.5220, -55.0121, -55.6212]],
       grad_fn=<SliceBackward0>)